In [1]:
import argparse
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, EarlyStoppingCallback, BitsAndBytesConfig, pipeline
from datasets import load_dataset, Dataset
from peft import LoraConfig, get_peft_model, PeftModel
from transformers import DataCollatorWithPadding
import evaluate
import numpy as np
from transformers import TrainerCallback
from torch.utils.data import DataLoader
from trl import SFTTrainer
from typing import Dict, Union, List, Any

/home/aerol1/miniconda3/envs/tda-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from peft import get_peft_model, LoraConfig, TaskType
def load_model(model_id="", device= None, use_lora:bool=False):
    
    device = device if device else "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_id,
                                            trust_remote_code=True,
                                            padding_side = "right")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.add_eos_token
    
    model_load_kwargs = {
    "trust_remote_code": True,
    "device_map": device,
    "torch_dtype":torch.bfloat16,
}
    model = AutoModelForCausalLM.from_pretrained(model_id, **model_load_kwargs)
    # Apply LoRA if specified
    if use_lora:
        lora_config = LoraConfig(
            r=8,
            lora_alpha=32,
            target_modules=["q_proj", "v_proj"],
            lora_dropout=0.1,
            bias="none",
            task_type=TaskType.CAUSAL_LM
        )
        model = get_peft_model(model, lora_config)
        model.print_trainable_parameters()
    
                                                
    return model, tokenizer

About Dataset:

* GSM8K consists of 8.5K high quality grade school math problems created by human problem writers. 
* Consists of "question" and "answer"

In [6]:
from datasets import load_dataset
data = load_dataset("google/IFEval")
data

DatasetDict({
    train: Dataset({
        features: ['key', 'prompt', 'instruction_id_list', 'kwargs'],
        num_rows: 541
    })
})

In [9]:
split_dataset = data["train"].train_test_split(test_size=0.05,seed=43)
train_split = split_dataset["train"].train_test_split(test_size=0.05,seed=43)
train_data = train_split["train"]
val_data = train_split["test"]
test_data = split_dataset['test']
train_data, val_data, test_data

(Dataset({
     features: ['key', 'prompt', 'instruction_id_list', 'kwargs'],
     num_rows: 487
 }),
 Dataset({
     features: ['key', 'prompt', 'instruction_id_list', 'kwargs'],
     num_rows: 26
 }),
 Dataset({
     features: ['key', 'prompt', 'instruction_id_list', 'kwargs'],
     num_rows: 28
 }))

In [10]:
index = 10
example = train_data[index]

prompt = example["prompt"]
instructions = example["instruction_id_list"]
kwargs = example["kwargs"]

print("🔸 Prompt:\n", prompt)
print("🔹 Instruction IDs:\n", instructions)
print("🔸 Constraints (kwargs):")
for i, kw in enumerate(kwargs):
    non_nulls = {k: v for k, v in kw.items() if v is not None}
    print(f"  - Constraint {i+1}:", non_nulls)


🔸 Prompt:
 Write a short fiction about adulthood. Make sure the word cousins appears more than 2 times.
🔹 Instruction IDs:
 ['keywords:frequency']
🔸 Constraints (kwargs):
  - Constraint 1: {'relation': 'at least', 'keyword': 'cousins', 'frequency': 3}


In [11]:
instruct_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
instruct_model, instruct_tokenizer = load_model(instruct_model_name, device="cuda:0")

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


In [12]:
def preprocess(example, tokenizer, add_instruction_ids=True, add_constraints=True):
    prompt = example["prompt"]
    instruction_ids = example.get("instruction_id_list", [])
    kwargs_list = example.get("kwargs", [])

    # Optionally format instruction IDs
    instruction_text = ""
    if add_instruction_ids and instruction_ids:
        instruction_text = "Instructions: " + ", ".join(instruction_ids) + "\n"

    # Optionally format kwargs (constraints)
    constraint_texts = []
    if add_constraints and kwargs_list:
        for i, kw in enumerate(kwargs_list):
            filtered = {k: v for k, v in kw.items() if v is not None}
            if filtered:
                constraint_texts.append(f"Constraint {i+1}: {filtered}")
    constraint_block = "\n".join(constraint_texts)
    
    # Final prompt text
    full_prompt = f"{instruction_text}{constraint_block}\n\n{prompt}".strip()

    # Tokenize prompt (input) and answer (target)
    q_ids = tokenizer(full_prompt, add_special_tokens=False)["input_ids"]
    a_ids = tokenizer(example["response"], add_special_tokens=False)["input_ids"]

    input_ids = q_ids + a_ids
    labels = [-100] * len(q_ids) + a_ids
    attention_mask = [1] * len(input_ids)

    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": attention_mask
    }


In [ ]:
from torch.nn.utils.rnn import pad_sequence

def custom_data_collator(features):
    # check torch.long
    input_ids = [torch.tensor(f["input_ids"], dtype=torch.long) for f in features]
    attention_mask = [torch.tensor(f["attention_mask"], dtype=torch.long) for f in features]
    labels = [torch.tensor(f["labels"], dtype=torch.long) for f in features]

    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels = pad_sequence(labels, batch_first=True, padding_value=-100)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

data_collator = custom_data_collator


DEFAULT_SYSTEM_PROMPT1 - High-Level Instruction Prompt
DEFAULT_SYSTEM_PROMPT2 - Step-by-Step Reasoning
DEFAULT_SYSTEM_PROMPT3 - Think Step-by-Step Prompt
DEFAULT_SYSTEM_PROMPT4 - Answer Validation Prompt
DEFAULT_SYSTEM_PROMPT5 - JSON-style Answer Output

In [ ]:
prompt_style = dict(DEFAULT_SYSTEM_PROMPT1 = """
You are a helpful writing assistant. Your job is to classify the constraints of a writing instruction.

You will be given:
- A text prompt
- A list of high-level instruction IDs
- A list of detailed constraint objects (called "keywords")

Your task is to output a list of keyword field names used in the constraints. These field names must come from a predefined set of 25 possible keys.
                    ⚠️ Output must be a list of dictionaries corresponding keywords with their values. Do not include explanations.

Here are the 25 possible field names:
["num_highlights", "relation", "num_words", "num_placeholders", "prompt_to_repeat", "num_bullets", "section_spliter", "num_sections", "capital_relation", "capital_frequency", "keywords", "num_paragraphs", "language", "let_relation", "letter", "let_frequency", "end_phrase", "forbidden_words", "keyword", "frequency", "num_sentences", "postscript_marker", "first_word", "nth_paragraph"]
---

Example:

Input: (Prompt + Instruction IDs)
                    
Prompt: Write a 300+ word summary of the Wikipedia page "https://en.wikipedia.org/wiki/Raymond_III,_Count_of_Tripoli". Do not use any commas and highlight at least 3 sections in markdown format.
Instruction IDs: [
  "punctuation:no_comma",
  "detectable_format:number_highlighted_sections",
  "length_constraints:number_words"
]

Output (Keywords):
Keywords:
[
  {"num_highlights": 3},
  {"relation": "at least", "num_words": 300}
]

""")

def create_prompt_deepseek_qwen(text: str, prompt_style: str) -> str:
    return f"<｜begin of sentence｜><｜User｜>{prompt_style}\\n{text.strip()}<｜Assistant｜>\n"

In [121]:
print(create_prompt_deepseek_qwen("John has 3 pens. He buys 2 more. How many pens?", prompt_style["DEFAULT_SYSTEM_PROMPT1"]))

<｜begin of sentence｜><｜User｜>You are a highly skilled math teacher. Your task is to solve the given grade-school level word problem. 
Carefully analyze the problem, perform all necessary intermediate steps, and provide a final numeric answer.
Present your reasoning in a clear, step-by-step manner followed by the final answer on the last line.

Format:
- Start with step-by-step explanation.
- End with: "Answer: [final numeric answer]"

Example:
Question: Mary has 3 apples. She buys 2 more and then eats 1. How many apples does she have?
Solution:
Mary starts with 3 apples.
She buys 2 more: 3 + 2 = 5.
She eats 1: 5 - 1 = 4.
Answer: 4
\nJohn has 3 pens. He buys 2 more. How many pens?<｜Assistant｜>



In [124]:
tokenized_dataset = train_data.map(
    lambda x: preprocess(x, instruct_tokenizer),
    remove_columns=["question", "answer"]
)

Map: 100%|██████████| 6352/6352 [00:03<00:00, 2078.75 examples/s]


In [125]:
tokenized_val = val_data.map(lambda x: preprocess(x, instruct_tokenizer), remove_columns=["question", "answer"])
tokenized_test = test_data.map(lambda x: preprocess(x, instruct_tokenizer), remove_columns=["question", "answer"])

Map: 100%|██████████| 1319/1319 [00:00<00:00, 2182.92 examples/s]


In [130]:
from torch.nn.utils.rnn import pad_sequence
import torch

def custom_data_collator(features):
    input_ids = [torch.tensor(f["input_ids"], dtype=torch.long) for f in features]
    attention_mask = [torch.tensor(f["attention_mask"], dtype=torch.long) for f in features]
    labels = [torch.tensor(f["labels"], dtype=torch.long) for f in features]

    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=instruct_tokenizer.pad_token_id)
    attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels = pad_sequence(labels, batch_first=True, padding_value=-100)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

# Use it
data_collator = custom_data_collator


In [131]:
sample_batch = tokenized_dataset.select(range(4))
sample_batch = [dict(x) for x in sample_batch]  # convert Dataset to list of dicts
collated = data_collator(sample_batch)


print("Keys:", collated.keys())
print("input_ids shape:", collated["input_ids"].shape)
print("labels shape:", collated["labels"].shape)
print("attention_mask shape:", collated["attention_mask"].shape)

# Optional: print a sample
print("Sample labels:\n", collated["labels"][0])


Keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids shape: torch.Size([4, 240])
labels shape: torch.Size([4, 240])
attention_mask shape: torch.Size([4, 240])
Sample labels:
 tensor([ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,    42,   587,
        39662,   594, 40358,   304,  6149,  2906,   572,   400,    23,   488,
          400,    17,   284,   400,  2442,    23,    10,    17,    28,    16,
           15,  2452,    16,    15,   624, 22816,   558,  1059,  6149,  2906,
        40358,   374,   400,    16,    15,   856,   220,    17,   284,   400,
         2442,    16,    15, 

In [141]:
for i, item in enumerate(sample_batch):
    print(f"Sample {i}: input_ids type: {type(item['input_ids'])}, length: {len(item['input_ids'])}")

Sample 0: input_ids type: <class 'list'>, length: 215
Sample 1: input_ids type: <class 'list'>, length: 137
Sample 2: input_ids type: <class 'list'>, length: 149
Sample 3: input_ids type: <class 'list'>, length: 240


In [142]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_dataset,
    shuffle=True,
    batch_size=8,
    collate_fn=data_collator
)


In [143]:
def evaluate(model, tokenizer, text, device, prompt_style):
    prompt = create_prompt_deepseek_qwen(text, prompt_style)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [135]:
# trying 5 run to see how response changes
# System prompt 1

for i in range(3):
    print(f"Run {i+1}\n")
    output = evaluate(instruct_model,instruct_tokenizer,question,device="cuda",prompt_style=prompt_style["DEFAULT_SYSTEM_PROMPT1"])
    print(output)
    print("____________________________________")
    print("____________________________________")

Run 1

<｜begin of sentence｜><｜User｜>You are a highly skilled math teacher. Your task is to solve the given grade-school level word problem. 
Carefully analyze the problem, perform all necessary intermediate steps, and provide a final numeric answer.
Present your reasoning in a clear, step-by-step manner followed by the final answer on the last line.

Format:
- Start with step-by-step explanation.
- End with: "Answer: [final numeric answer]"

Example:
Question: Mary has 3 apples. She buys 2 more and then eats 1. How many apples does she have?
Solution:
Mary starts with 3 apples.
She buys 2 more: 3 + 2 = 5.
She eats 1: 5 - 1 = 4.
Answer: 4
\nSusan had a bouquet of 3 dozen roses.  She gave half to her daughter, and then placed the rest in a vase.  The next day, one-third of the flowers in the vase were wilted.  After removing the wilted flowers, how many flowers remained in the vase?<｜Assistant｜>
Let's break down this problem step-by-step:

Step 1: Determine the total number of roses Susa

In [136]:
# System prompt 2

for i in range(3):
    print(f"Run {i+1}\n")
    output = evaluate(instruct_model,instruct_tokenizer,question,device="cuda",prompt_style=prompt_style["DEFAULT_SYSTEM_PROMPT2"])
    print(output)
    print("____________________________________")
    print("____________________________________")

Run 1

<｜begin of sentence｜><｜User｜>You are a math tutor helping students with word problems. Follow these instructions:
1. Read the question carefully.
2. Write down all numerical values and units.
3. Break down the operations in order.
4. Provide final answer in format: "Answer: [number]"

If needed, round to the nearest whole number or decimal as appropriate.

Example:
Question: Sarah read 12 books last year. This year, she read 4 more than twice that number. How many books did she read this year?
Solution:
Last year: 12 books.
This year: 2 × 12 = 24. Then add 4 more: 24 + 4 = 28.
Answer: 28
\nSusan had a bouquet of 3 dozen roses.  She gave half to her daughter, and then placed the rest in a vase.  The next day, one-third of the flowers in the vase were wilted.  After removing the wilted flowers, how many flowers remained in the vase?<｜Assistant｜>
Let's break down the problem step by step:

1. **Read the question carefully:** Susan had a bouquet of 3 dozen roses. We need to determin

In [137]:
# System prompt 3

for i in range(3):
    print(f"Run {i+1}\n")
    output = evaluate(instruct_model,instruct_tokenizer,question,device="cuda",prompt_style=prompt_style["DEFAULT_SYSTEM_PROMPT3"])
    print(output)
    print("____________________________________")
    print("____________________________________")

Run 1

<｜begin of sentence｜><｜User｜>You are a thoughtful problem solver. Solve the math problem below step-by-step.
Do not jump to the answer immediately.
Make each operation clear and explain your reasoning at every stage.

End your response with:
Answer: [final number]

Example:
Question: Tom has 5 red balls and 7 blue balls. He gives away 3 red and 2 blue balls. How many does he have now?
Solution:
Tom starts with 5 red and 7 blue balls.
He gives away 3 red: 5 - 3 = 2.
He gives away 2 blue: 7 - 2 = 5.
Total left: 2 + 5 = 7.
Answer: 7
\nSusan had a bouquet of 3 dozen roses.  She gave half to her daughter, and then placed the rest in a vase.  The next day, one-third of the flowers in the vase were wilted.  After removing the wilted flowers, how many flowers remained in the vase?<｜Assistant｜>
To solve this problem, let's break it down into steps:

1. **Determine the total number of roses Susan initially had:**
   Susan had 3 dozen roses. Since there are 12 roses in a dozen, she had \(3

In [138]:
# System prompt 4

for i in range(3):
    print(f"Run {i+1}\n")
    output = evaluate(instruct_model,instruct_tokenizer,question,device="cuda",prompt_style=prompt_style["DEFAULT_SYSTEM_PROMPT4"])
    print(output)
    print("____________________________________")
    print("____________________________________")

Run 1

<｜begin of sentence｜><｜User｜>You're a math assistant. Solve the question below with full explanation.
- Include all necessary units
- Justify each arithmetic step
- The last line must be: "Answer: [final number]"

If there's more than one step, show them all.

Example:
Question: A farmer has 10 cows and 15 chickens. Each cow eats 5 kg of hay per day, and each chicken eats 1 kg. How much total hay is needed per day?
Solution:
10 cows × 5 kg = 50 kg
15 chickens × 1 kg = 15 kg
Total = 50 + 15 = 65 kg
Answer: 65
\nSusan had a bouquet of 3 dozen roses.  She gave half to her daughter, and then placed the rest in a vase.  The next day, one-third of the flowers in the vase were wilted.  After removing the wilted flowers, how many flowers remained in the vase?<｜Assistant｜>
Let's break down the problem into steps:

### Step 1: Determine the initial number of roses

Susan initially has 3 dozen roses. Since one dozen equals 12, we calculate:

\[ \text{Initial number of roses} = 3 \times 12 

In [139]:
# System prompt 5

for i in range(3):
    print(f"Run {i+1}\n")
    output = evaluate(instruct_model,instruct_tokenizer,question,device="cuda",prompt_style=prompt_style["DEFAULT_SYSTEM_PROMPT5"])
    print(output)
    print("____________________________________")
    print("____________________________________")

Run 1



<｜begin of sentence｜><｜User｜>You're a math-solving assistant. For the given problem:
1. Explain the reasoning.
2. Show all steps.
3. Provide final answer in this JSON format:

{"answer": [NUMBER]}

Example:
Question: A box contains 6 red and 4 green balls. If 2 red and 1 green are removed, how many are left?
Solution:
Start with 6 red + 4 green = 10 balls.
Remove 2 red and 1 green: 10 - 3 = 7.
{"answer": 7}
\nSusan had a bouquet of 3 dozen roses.  She gave half to her daughter, and then placed the rest in a vase.  The next day, one-third of the flowers in the vase were wilted.  After removing the wilted flowers, how many flowers remained in the vase?<｜Assistant｜>
To solve this problem, let's break it down into steps.

### Step 1: Calculate the total number of roses Susan initially has
- Susan starts with a bouquet of 3 dozen roses.
- One dozen is equal to 12 roses.
- Therefore, 3 dozen roses = \( 3 \times 12 = 36 \) roses.

### Step 2: Determine how many roses Susan gives to her daught

In [144]:
# checking if padding is properly done, and eos is replaced with -100 
collated_batch = data_collator(sample_batch)

print("Collated Batch Keys:", collated_batch.keys())

if "input_ids" in collated_batch:
    print("Input IDs Shape:", collated_batch["input_ids"].shape)
    print("Example Input IDs:\n", collated_batch["input_ids"])

if "attention_mask" in collated_batch:
    print("Attention Mask Shape:", collated_batch["attention_mask"].shape)
    print("Example Attention Mask:\n", collated_batch["attention_mask"])

if "labels" in collated_batch:
    print("Labels Shape:", collated_batch["labels"].shape)
    print("Example Labels:\n", collated_batch["labels"])

Collated Batch Keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Input IDs Shape: torch.Size([4, 240])
Example Input IDs:
 tensor([[    42,    587,  39662,  21189,    264,  17059,  40358,    369,  26568,
            678,    315,   1059,  81469,     13,  11954,   6149,   2906,     11,
           1059,  40358,    572,    400,     17,    803,   1091,    400,     23,
            714,   2337,   1059,   9990,   1042,     11,    220,   1059,  40358,
            572,    400,     20,    803,   1091,  10917,   1059,   6149,   2906,
          40358,     13,   3555,    374,    279,  11414,   5263,    304,  64063,
            594,  17059,  40358,     30,     42,    587,  39662,    594,  40358,
            304,   6149,   2906,    572,    400,     23,    488,    400,     17,
            284,    400,   2442,     23,     10,     17,     28,     16,     15,
           2452,     16,     15,    624,  22816,    558,   1059,   6149,   2906,
          40358,    374,    400,     16,     15,    856,  

In [146]:
# Check if tokenization aligns question and answer properly
from pprint import pprint

for i, feature in enumerate(sample_batch):
    print(f"\n🔍 Sample {i+1}")
    print("🟠 Original Question:", train_data[i]["question"])
    print("🟢 Original Answer:", train_data[i]["answer"])

    decoded_input = instruct_tokenizer.decode(feature["input_ids"], skip_special_tokens=False)
    print("🔵 Decoded Input IDs:\n", decoded_input)

    # Find where the labels are non -100
    label_ids = feature["labels"]
    answer_token_ids = [id for id in label_ids if id != -100]
    decoded_answer = instruct_tokenizer.decode(answer_token_ids, skip_special_tokens=True)
    print("🟣 Decoded Labels (Answer Tokens Only):\n", decoded_answer)



🔍 Sample 1
🟠 Original Question: Kathleen receives a weekly allowance for completing all of her chores. During middle school, her allowance was $2 more than $8 but during her senior year,  her allowance was $5 more than twice her middle school allowance. What is the percentage increase in Kathleen's weekly allowance?
🟢 Original Answer: Kathleen's allowance in middle school was $8 + $2 = $<<8+2=10>>10.
Twice her middle school allowance is $10 x 2 = $<<10*2=20>>20.
So, Kathleen's senior year allowance is $20 + $5 = $<<20+5=25>>25.
Her allowance increased by $25 - $10 = $<<25-10=15>>15.
Kathleen's percent increase in her weekly allowance is $15/$10 x 100% = <<15/10*100=150>>150%.
#### 150
🔵 Decoded Input IDs:
 Kathleen receives a weekly allowance for completing all of her chores. During middle school, her allowance was $2 more than $8 but during her senior year,  her allowance was $5 more than twice her middle school allowance. What is the percentage increase in Kathleen's weekly allowanc